## 데이터 로드

In [ ]:
from datasets import Dataset
from transformers import BertForTokenClassification, AutoTokenizer
import torch

In [ ]:
data = [
    {
        "input": "철수는 서울에 간다.",
        "label": [
            "B-PER",      # 철수는
            "B-LOC",      # 서울에
            "O",          # 간다.
        ]
    },
    {
        "input": "삼성은 어제 발표했다.",
        "label": [
            "B-ORG",      # 삼성은
            "B-DATE",     # 어제
            "O",          # 발표했다.
        ]
    },
    {
        "input": "부산에서 축구를 한다.",
        "label": [
            "B-LOC",      # 부산에서
            "B-MISC",     # 축구를
            "O",          # 한다.
        ]
    }
]

In [ ]:
label_to_id = {
    "O": 0,         # 개체명 아님
    "B-PER": 1,     # 사람 (시작)
    "I-PER": 2,     # 사람 (내부)
    "B-LOC": 3,     # 위치 (시작)
    "I-LOC": 4,     # 위치 (내부)
    "B-ORG": 5,     # 기관/단체 (시작)
    "I-ORG": 6,     # 기관/단체 (내부)
    "B-DATE": 7,    # 날짜/시간 (시작)
    "I-DATE": 8,    # 날짜/시간 (내부)
    "B-MISC": 9,    # 기타 (시작)
    "I-MISC": 10    # 기타 (내부)
}

In [ ]:
id_to_label = {
    0: "O",
    1: "B-PER",
    2: "I-PER",
    3: "B-LOC",
    4: "I-LOC",
    5: "B-ORG",
    6: "I-ORG",
    7: "B-DATE",
    8: "I-DATE",
    9: "B-MISC",
    10: "I-MISC"
}

In [ ]:
label_list = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-DATE', 'I-DATE', 'B-MISC', 'I-MISC']

In [ ]:
dataset = Dataset.from_list(data)
dataset

Dataset({
    features: ['input', 'label'],
    num_rows: 3
})

In [ ]:
dataset['input']

Column(['철수는 서울에 간다.', '삼성은 어제 발표했다.', '부산에서 축구를 한다.'])

In [ ]:
dataset[0]

{'input': '철수는 서울에 간다.', 'label': ['B-PER', 'B-LOC', 'O']}

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", use_fast=True)

## 토크나이저(dataset.map)

In [ ]:
label_list = ['O', 'B-PER', 'I-PER', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-DATE', 'I-DATE', 'B-MISC', 'I-MISC']
def build_tag_map(all_labels):
    label2id = {}
    id2label = {}
    for idx, label in enumerate(all_labels):
        label2id[label] = idx
        id2label[idx] = label
    return label2id, id2label
    return None, None
label2id, id2label = build_tag_map(label_list)

In [ ]:
id2label, label2id

({0: 'O',
  1: 'B-PER',
  2: 'I-PER',
  3: 'B-LOC',
  4: 'I-LOC',
  5: 'B-ORG',
  6: 'I-ORG',
  7: 'B-DATE',
  8: 'I-DATE',
  9: 'B-MISC',
  10: 'I-MISC'},
 {'O': 0,
  'B-PER': 1,
  'I-PER': 2,
  'B-LOC': 3,
  'I-LOC': 4,
  'B-ORG': 5,
  'I-ORG': 6,
  'B-DATE': 7,
  'I-DATE': 8,
  'B-MISC': 9,
  'I-MISC': 10})

In [ ]:
len(id2label)

11

### Data split

In [ ]:
def split_input(batch):
    return {"splited_input": [s.split() for s in batch["input"]]}

### label to ids

In [ ]:
def label2ids(batch):
    if isinstance(batch["label"][0][0], str):
        return {"labels_ids": [[label2id[t] for t in seq] for seq in batch["label"]]}
    return {"labels_ids": batch["label"]}

In [ ]:
dataset = dataset.map(split_input, batched=True, remove_columns='input')
dataset = dataset.map(label2ids, batched=True, remove_columns='label')
dataset

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Dataset({
    features: ['splited_input', 'labels_ids'],
    num_rows: 3
})

### 토큰화, 라벨 확장

In [ ]:
# 5) 토큰화 + 라벨 정렬 (첫 서브워드만 라벨, 나머지는 -100)
def tokenize_and_extend_labels(batch, label_all_tokens=False):
    tokenized_input = tokenizer(batch["splited_input"], is_split_into_words=True, truncation=True)
    aligned = []
    for i, word_labels in enumerate(batch["labels_ids"]):
        word_ids = tokenized_input.word_ids(batch_index=i)
        prev = None
        label_ids = []
        for w_id in word_ids:
            if w_id is None:
                label_ids.append(-100)
            elif w_id != prev:
                label_ids.append(word_labels[w_id])
            else:
                label_ids.append(word_labels[w_id] if label_all_tokens else -100)
            prev = w_id
        aligned.append(label_ids)
    tokenized_input["labels"] = aligned
    return tokenized_input

In [ ]:
dataset = dataset.map(split_input, batched=True, remove_columns='input')
dataset = dataset.map(label2ids, batched=True, remove_columns='label')
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True, remove_columns=dataset.column_names)
tokenized_dataset

ValueError: Column to remove ['input'] not in the dataset. Current columns in the dataset: ['splited_input', 'labels_ids']

In [ ]:
tokenized_dataset['input_ids']

In [ ]:
for t in tokenized_dataset['input_ids']:
  decode = tokenizer.convert_ids_to_tokens(t)
  print(decode)

In [ ]:
tokenized_dataset['labels']

In [ ]:
for input, label_list in zip(tokenized_dataset['input_ids'], tokenized_dataset['labels']):
  label = t.tolist()
  arr = []
  for l in label:
    if l == -100:
      arr.append(l)
      continue

    arr.append(id2label[l])
  decode = tokenizer.convert_ids_to_tokens(input)
  print()
  print(decode)
  print(arr)

## DataLoader, DataCollatorForTokenClassification


In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
train_dataloader = DataLoader(
    tokenized_dataset, shuffle=True, batch_size=2, collate_fn=data_collator
)

### dataloder 테스트

In [ ]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

In [ ]:
batch = {k: v.to(device) for k, v in batch.items()}

In [ ]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

## 학습 모델, 옵티마이저, 에폭 설정

In [ ]:
from transformers import get_scheduler
from torch.optim import AdamW
from transformers import DataCollatorForTokenClassification

model = BertForTokenClassification.from_pretrained("klue/bert-base", num_labels=len(id2label), id2label=id2label, label2id=label2id).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=int(0.1 * num_training_steps), num_training_steps=num_training_steps)

## 학습

In [ ]:
num_epochs = 3

for epoch in range(num_epochs):
    total_loss = one_epoch_train(model, train_dataloader, optimizer, scheduler)

### 학습 함수

In [ ]:
def one_epoch_train(model, train_dataloader, optimizer, scheduler):
  model.train()
  total_loss = 0.0
  for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)       # out.loss, out.logits
        loss = out.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()

  print(f"epoch {epoch+1} train_loss={total_loss/len(train_dataloader):.4f}")
  return total_loss/len(train_dataloader)

## 추론

In [ ]:
label_to_slot = {
    "B-PER": '사람',     # 사람 (시작)
    "B-LOC": '위치',     # 위치 (시작)
    "B-ORG": '기관',     # 기관/단체 (시작)
    "B-DATE": '날짜',    # 날짜/시간 (시작)
    "B-MISC": '기타',    # 기타 (시작)
    "-100": "NONE"
}

In [ ]:
label_to_all_slot = {
    "B-PER": '사람',     # 사람 (시작)
    "I-PER": '사람',     # 사람 (내부)
    "B-LOC": '위치',     # 위치 (시작)
    "I-LOC": '위치',     # 위치 (내부)
    "B-ORG": '기관',     # 기관/단체 (시작)
    "I-ORG": '기관',     # 기관/단체 (내부)
    "B-DATE": '날짜,    # 날짜/시간 (시작)
    "I-DATE": '날짜',    # 날짜/시간 (내부)
    "B-MISC": '기타',    # 기타 (시작)
    "I-MISC": '기타'    # 기타 (내부)
}

In [ ]:
def inference(sentences):
  text = sentences.split()
  tokenized_input = tokenizer(text, return_tensors="pt", is_split_into_words=True, truncation=True)
  input = {k: v.to(device) for k, v in tokenized_input.items()}
  with torch.no_grad():
      logits = model(**input).logits

  predictions = torch.argmax(logits, dim=2)
  predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]

  decode = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'][0])

  for word, pred in zip(decode, predicted_token_class):
    if label_to_slot.get(pred):
      print(f'{label_to_slot[pred]}: ', word)

In [ ]:
inference('철수는 서울에 간다.')

## 최종 코드

## BERT

In [ ]:
import json
from datasets import Dataset
from transformers import DataCollatorForTokenClassification, get_scheduler, BertForTokenClassification, AutoTokenizer
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

label_list = [
    'O',
    "B-JOB", # 직무
    "I-JOB",
    "B-CAR", # 경력
    "I-CAR",
    "B-EDU", # 학력
    "I-EDU",
    "B-LOC", # 지역
    "I-LOC"
]

with open('/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/url_llm_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", use_fast=True)

def build_tag_map(all_labels):
    label2id = {}
    id2label = {}
    for idx, label in enumerate(all_labels):
        label2id[label] = idx
        id2label[idx] = label
    return label2id, id2label
    return None, None

def label2ids(batch):
    if isinstance(batch["label"][0][0], str):
        return {"labels_ids": [[label2id[t] for t in seq] for seq in batch["label"]]}
    return {"labels_ids": batch["label"]}

# 5) 토큰화 + 라벨 정렬 (첫 서브워드만 라벨, 나머지는 -100)
def tokenize_and_extend_labels(batch):
    tokenized_input = tokenizer(batch["input"], truncation=True)
    aligned = []

    for word_labels in batch["labels_ids"]:
      label_ids = [0] + word_labels + [0]
      aligned.append(label_ids)

    tokenized_input["labels"] = aligned
    return tokenized_input

label2id, id2label = build_tag_map(label_list)

dataset = dataset.map(label2ids, batched=True, remove_columns='label')
tokenized_dataset = dataset.map(tokenize_and_extend_labels, batched=True,
                                remove_columns=dataset.column_names,
                                )

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
train_dataloader = DataLoader(
    tokenized_dataset, shuffle=True, batch_size=2, collate_fn=data_collator
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = BertForTokenClassification.from_pretrained("klue/bert-base", num_labels=len(id2label), id2label=id2label, label2id=label2id).to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 50

num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=int(0.1 * num_training_steps), num_training_steps=num_training_steps)

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in train_dataloader:
          batch = {k: v.to(device) for k, v in batch.items()}
          out = model(**batch)       # out.loss, out.logits
          loss = out.loss
          loss.backward()

          torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
          optimizer.step()
          scheduler.step()
          optimizer.zero_grad(set_to_none=True)

          total_loss += loss.item()

    print(f"epoch {epoch+1} train_loss={total_loss/len(train_dataloader):.4f}")

### 평가코드


In [ ]:
import json
from datasets import Dataset
from transformers import DataCollatorForTokenClassification, get_scheduler, BertForTokenClassification, AutoTokenizer
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# ============================================================
# 1. 라벨 리스트 정의
# ============================================================
label_list = [
    'O',
    "B-JOB",  # 직무
    "I-JOB",
    "B-CAR",  # 경력
    "I-CAR",
    "B-EDU",  # 학력
    "I-EDU",
    "B-LOC",  # 지역
    "I-LOC"
]

# ============================================================
# 2. 데이터 로드
# ============================================================
with open('/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/url_llm_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

dataset = Dataset.from_list(data)

# ============================================================
# 3. Train / Eval 분리
# ============================================================
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

# ============================================================
# 4. 라벨 매핑 생성
# ============================================================
def build_tag_map(all_labels):
    label2id = {}
    id2label = {}
    for idx, label in enumerate(all_labels):
        label2id[label] = idx
        id2label[idx] = label
    return label2id, id2label

label2id, id2label = build_tag_map(label_list)

# ============================================================
# 5. 라벨을 id로 바꾸는 함수
# ============================================================
def label2ids(batch):
    # batch["label"]: [['O','B-JOB','I-JOB',...], ...]
    if isinstance(batch["label"][0][0], str):
        return {"labels_ids": [[label2id[t] for t in seq] for seq in batch["label"]]}
    return {"labels_ids": batch["label"]}

# ============================================================
# 6. 토큰화 함수
# ============================================================
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", use_fast=True)

def tokenize_and_extend_labels(batch):
    tokenized_input = tokenizer(batch["input"], truncation=True)
    aligned_labels = []
    for label_seq in batch["labels_ids"]:
        label_ids = [0] + label_seq + [0]
        aligned_labels.append(label_ids)
    tokenized_input["labels"] = aligned_labels
    return tokenized_input

# ============================================================
# 7. 전처리 적용
# ============================================================
train_dataset = train_dataset.map(label2ids, batched=True, remove_columns='label')
eval_dataset = eval_dataset.map(label2ids, batched=True, remove_columns='label')

train_dataset = train_dataset.map(tokenize_and_extend_labels, batched=True, remove_columns=train_dataset.column_names)
eval_dataset = eval_dataset.map(tokenize_and_extend_labels, batched=True, remove_columns=eval_dataset.column_names)

# ============================================================
# 8. DataLoader 생성
# ============================================================
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=2, collate_fn=data_collator)
eval_dataloader = DataLoader(eval_dataset, shuffle=False, batch_size=2, collate_fn=data_collator)

# ============================================================
# 9. 모델 및 옵티마이저 설정
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
model = BertForTokenClassification.from_pretrained(
    "klue/bert-base",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
).to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

# ============================================================
# 10. 학습 및 평가 루프
# ============================================================
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    # --------------------
    # (1) 학습 루프
    # --------------------
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        loss = out.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"[Epoch {epoch+1}] Train Loss = {avg_train_loss:.4f}")

    # --------------------
    # (2) 평가 루프
    # --------------------
    model.eval()
    eval_loss = 0.0
    preds_list = []
    labels_list = []

    with torch.no_grad():
        for batch in eval_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)

            loss = out.loss
            eval_loss += loss.item()

            logits = out.logits  # (batch, seq_len, num_labels)
            predictions = torch.argmax(logits, dim=-1)  # 예측된 라벨 id

            # 실제 라벨
            labels = batch["labels"]

            # padding(-100) 제거하고 평탄화
            for i in range(labels.shape[0]):
                for j in range(labels.shape[1]):
                    if labels[i, j] != -100:
                        preds_list.append(predictions[i, j].item())
                        labels_list.append(labels[i, j].item())

    avg_eval_loss = eval_loss / len(eval_dataloader)
    print(f"[Epoch {epoch+1}] Eval Loss = {avg_eval_loss:.4f}")

    # --------------------
    # (3) 평가 지표 출력
    # --------------------
    print("\n=== Classification Report ===")
    label_names = [id2label[i] for i in range(len(id2label))]
    print(classification_report(labels_list, preds_list, target_names=label_names, zero_division=0))
    print("========================================\n")


### 모델 저장

In [ ]:
save_path = '/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/bert_model.pt'
torch.save(model, save_path)

### 추론

In [ ]:
def inference(sentences):
  label_to_slot = {
    'O': "O",
    "B-JOB" : "직무", # 직무
    "I-JOB": "직무",
    "B-CAR": "경력", # 경력
    "I-CAR": "경력",
    "B-EDU": "학력", # 학력
    "I-EDU": "학력",
    "B-LOC": "지역", # 지역
    "I-LOC": "지역"
  }
  label2id, id2label = build_tag_map(label_list)

  tokenized_input = tokenizer(sentences, return_tensors="pt", truncation=True)
  input = {k: v.to(device) for k, v in tokenized_input.items()}
  with torch.no_grad():
      logits = model(**input).logits

  predictions = torch.argmax(logits, dim=2)
  predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]

  decode = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'][0])

  res = {
      '직무': '',
      '경력': '',
      '학력': '',
      '지역': ''
  }

  for word, pred in zip(decode, predicted_token_class):
    word = word.replace('#', '')
    if label_to_slot.get(pred):
      print(f'{label_to_slot[pred]}: ', word)
      if label_to_slot[pred] != 'O':
          res[label_to_slot[pred]] += word

  return res

In [ ]:
inference("야, ERP 컨설턴트 경력 5년 대졸이면, 경기 공고 좋은 거 없냐?")

O:  [CLS]
O:  야
O:  ,
직무:  ER
직무:  P
직무:  컨설턴트
경력:  경력
경력:  5
경력:  년
학력:  대졸
O:  이면
O:  ,
지역:  경기
O:  공고
O:  좋
O:  은
O:  거
O:  없
O:  냐
O:  ?
O:  [SEP]


{'직무': 'ERP컨설턴트', '경력': '경력5년', '학력': '대졸', '지역': '경기'}

## CRF + BERT

In [ ]:
pip install -q pytorch-crf

### 유틸 함수

In [ ]:
def build_tag_map(all_labels):
    label2id = {}
    id2label = {}
    for idx, label in enumerate(all_labels):
        label2id[label] = idx
        id2label[idx] = label
    return label2id, id2label
    return None, None

def label2ids(batch):
    if isinstance(batch["label"][0][0], str):
        return {"labels_ids": [[label2id[t] for t in seq] for seq in batch["label"]]}
    return {"labels_ids": batch["label"]}

### 학습 함수

In [ ]:
from torchcrf import CRF
import json
from datasets import Dataset
from transformers import BertForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification, get_scheduler
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

def train_crf_bert(data_path, model_name):
    label_list = [
        'O',
        "B-JOB", # 직무
        "I-JOB",
        "B-CAR", # 경력
        "I-CAR",
        "B-EDU", # 학력
        "I-EDU",
        "B-LOC", # 지역
        "I-LOC"
    ]

    with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    dataset = Dataset.from_list(data)
    tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", use_fast=True)

    # 5) 토큰화 + 라벨 정렬 (첫 서브워드만 라벨, 나머지는 -100)
    def tokenize_and_extend_labels(batch):
        tokenized_input = tokenizer(batch["input"], truncation=True)
        aligned = []

        for word_labels in batch["labels_ids"]:
          label_ids = [0] + word_labels + [0]
          aligned.append(label_ids)

        tokenized_input["labels"] = aligned
        return tokenized_input

    label2id, id2label = build_tag_map(label_list)

    dataset = dataset.map(label2ids, batched=True, remove_columns='label')
    tokenized_dataset = dataset.map(tokenize_and_extend_labels, batched=True,
                                    remove_columns=dataset.column_names,
                                    )

    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    train_dataloader = DataLoader(
        tokenized_dataset, shuffle=True, batch_size=2, collate_fn=data_collator
    )

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = BertForTokenClassification.from_pretrained(model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id).to(device)
    crf = CRF(len(id2label), batch_first=True).to(device)

    optimizer = AdamW(list(model.parameters()) + list(crf.parameters()), lr=5e-5)

    num_epochs = 30
    num_training_steps = num_epochs * len(train_dataloader)
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=int(0.1 * num_training_steps), num_training_steps=num_training_steps)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        for batch in train_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)       # out.loss, out.logits
            logits = out.logits

            # 토크나이저/데이터콜레이터는 padding 토큰에 -100을 할당한다. CRF는 -100을 허용하지 않으므로 대체한다.
            labels_for_crf = batch["labels"].clone()                    # 원본 라벨 복제
            labels_for_crf[labels_for_crf == -100] = 0                  # 패딩(-100)을 'O' 인덱스(여기서는 0)로 대체

            # attention_mask를 bool으로 변환 (torchcrf는 bool mask를 기대)
            mask = batch["attention_mask"].bool()

            # CRF 손실 (음의 로그우도)
            loss = -crf(logits, labels_for_crf, mask=mask, reduction="mean")

            loss.backward()
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        print(f"epoch {epoch+1} train_loss={total_loss/len(train_dataloader):.4f}")

data_path = '/content/drive/MyDrive/ai_enginner/job_search/AI/data/url_exchanger/url_llm_data.json'
model_name = "klue/bert-base"
train_crf_bert(data_path, model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

Map:   0%|          | 0/238 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


epoch 1 train_loss=20.9914
epoch 2 train_loss=3.9743
epoch 3 train_loss=1.5423
epoch 4 train_loss=0.8952
epoch 5 train_loss=0.6385
epoch 6 train_loss=0.6508
epoch 7 train_loss=0.4569
epoch 8 train_loss=0.4049
epoch 9 train_loss=0.4540
epoch 10 train_loss=0.2857
epoch 11 train_loss=0.2548
epoch 12 train_loss=0.1572
epoch 13 train_loss=0.1118
epoch 14 train_loss=0.1199
epoch 15 train_loss=0.0495
epoch 16 train_loss=0.0256
epoch 17 train_loss=0.0192
epoch 18 train_loss=0.0191
epoch 19 train_loss=0.0159
epoch 20 train_loss=0.0175
epoch 21 train_loss=0.0497
epoch 22 train_loss=0.0308
epoch 23 train_loss=0.0322
epoch 24 train_loss=0.0136
epoch 25 train_loss=0.0117
epoch 26 train_loss=0.0122
epoch 27 train_loss=0.0098
epoch 28 train_loss=0.0283
epoch 29 train_loss=0.0139
epoch 30 train_loss=0.0106


### 모델 저장

In [ ]:
def save_model(save_dir):
    # 1) BERT 저장
    model.save_pretrained(f"{save_dir}/bert")

    # 2) 토크나이저 저장
    tokenizer.save_pretrained(f"{save_dir}/tokenizer")

    # 3) CRF는 HuggingFace 포맷이 아니므로 torch로 직접 저장
    torch.save(crf.state_dict(), f"{save_dir}/crf.pt")

    print('모델, 토크나이저, CRF 저장완료!: ', save_dir)

save_path = "/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/checkpoint"
save_model(save_path)

### 모델 로드

In [ ]:
from transformers import BertForTokenClassification, AutoTokenizer
from torchcrf import CRF
import torch

def load_model(load_dir):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # 1) 토크나이저, BERT 복원
    tokenizer = AutoTokenizer.from_pretrained(f"{load_dir}/tokenizer", use_fast=True)
    model = BertForTokenClassification.from_pretrained(f"{load_dir}/bert", num_labels=len(id2label), id2label=id2label, label2id=label2id).to(device)

    # 2) CRF 복원 (num_labels 일치해야 함)
    num_labels = model.config.num_labels
    crf = CRF(num_labels, batch_first=True)
    crf.load_state_dict(torch.load(f"{load_dir}/crf.pt", map_location=device))
    crf.to(device)

    return crf, model, tokenizer

crf, model, tokenizer = load_model("/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/checkpoint")

### 추론

In [ ]:
def inference_with_crf(sentence):
    label_to_slot = {
      'O': "O",
      "B-JOB" : "직무", # 직무
      "I-JOB": "직무",
      "B-CAR": "경력", # 경력
      "I-CAR": "경력",
      "B-EDU": "학력", # 학력
      "I-EDU": "학력",
      "B-LOC": "지역", # 지역
      "I-LOC": "지역"
    }

    tokenized_input = tokenizer(sentence, return_tensors="pt", truncation=True)
    input_data = {k: v.to(device) for k, v in tokenized_input.items()}

    with torch.no_grad():
        # BERT에서 logits 추출 (기존과 동일)
        logits = model(**input_data).logits  # [1, seq_len, num_labels]

        # CRF 디코딩 (argmax 대신 사용)
        predictions = crf.decode(logits)[0]  # 첫 번째 시퀀스 결과

        # 예측 라벨을 문자열로 변환
        predicted_token_class = [model.config.id2label[t] for t in predictions]

    # 토큰 디코딩 (기존과 동일)
    decode = tokenizer.convert_ids_to_tokens(tokenized_input['input_ids'][0])

    res = {
        '직무': '',
        '경력': '',
        '학력': '',
        '지역': ''
    }

    for word, pred in zip(decode, predicted_token_class):
      word = word.replace('#', '')
      if label_to_slot.get(pred):
        if label_to_slot[pred] != 'O':
            res[label_to_slot[pred]] += word

    return res

# 사용법 (기존과 동일)
res = inference_with_crf('야, ERP 컨설턴트 경력 5년 대졸이면, 경기 공고 좋은 거 없냐?')
res

{'직무': 'ERP컨설턴트', '경력': '경력5년', '학력': '대졸', '지역': '경기'}

### 평가코드

In [ ]:
from torchcrf import CRF
import json
from datasets import Dataset
from transformers import BertForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification, get_scheduler
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

# 1. 라벨 정의
label_list = [
    'O',
    "B-JOB", # 직무
    "I-JOB",
    "B-CAR", # 경력
    "I-CAR",
    "B-EDU", # 학력
    "I-EDU",
    "B-LOC", # 지역
    "I-LOC"
]

# 2. 데이터 로드
with open('/content/drive/MyDrive/ai_enginner/job_search/AI/src/url_exchanger/url_llm_data.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Dataset으로 변환
dataset = Dataset.from_list(data)

# 3. train / eval 나누기
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

# 4. 라벨 매핑 함수
def build_tag_map(all_labels):
    label2id = {}
    id2label = {}
    for idx, label in enumerate(all_labels):
        label2id[label] = idx
        id2label[idx] = label
    return label2id, id2label

# 5. 문자열 라벨을 숫자 라벨로 바꾸는 함수
def label2ids(batch):
    if isinstance(batch["label"][0][0], str):
        return {"labels_ids": [[label2id[t] for t in seq] for seq in batch["label"]]}
    return {"labels_ids": batch["label"]}

# 6. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", use_fast=True)

# 7. 토큰화 함수
def tokenize_and_extend_labels(batch):
    # tokenized_input: {'input_ids': [...], 'attention_mask': [...]}
    tokenized_input = tokenizer(batch["input"], truncation=True)
    aligned = []
    for word_labels in batch["labels_ids"]:
        # [CLS]와 [SEP]에 'O' 추가 (id=0)
        label_ids = [0] + word_labels + [0]
    tokenized_input["labels"] = aligned
    return tokenized_input

# 8. 라벨 매핑 생성
label2id, id2label = build_tag_map(label_list)

# 9. 매핑 적용
train_dataset = train_dataset.map(label2ids, batched=True, remove_columns='label')
eval_dataset = eval_dataset.map(label2ids, batched=True, remove_columns='label')

# 10. 토큰화 적용
train_dataset = train_dataset.map(tokenize_and_extend_labels, batched=True)
eval_dataset = eval_dataset.map(tokenize_and_extend_labels, batched=True)

# 11. 데이터 콜레이터 정의
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=2, collate_fn=data_collator)
eval_dataloader = DataLoader(eval_dataset, shuffle=False, batch_size=2, collate_fn=data_collator)

# 12. 모델 및 CRF 정의
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BertForTokenClassification.from_pretrained(
    "klue/bert-base",
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
).to(device)
crf = CRF(len(id2label), batch_first=True).to(device)

# 13. 옵티마이저 및 스케줄러 정의
optimizer = AdamW(list(model.parameters()) + list(crf.parameters()), lr=5e-5)
num_epochs = 5
num_training_steps = num_epochs * len(train_dataloader)
scheduler = get_scheduler("linear", optimizer=optimizer,
                          num_warmup_steps=int(0.1 * num_training_steps),
                          num_training_steps=num_training_steps)

# ===================================================================
# 학습 + 평가 루프
# ===================================================================

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        logits = out.logits

        # torchcrf는 -100 불가 -> 0('O')으로 치환
        labels_for_crf = batch["labels"].clone()
        labels_for_crf[labels_for_crf == -100] = 0
        mask = batch["attention_mask"].bool()

        loss = -crf(logits, labels_for_crf, mask=mask, reduction="mean")

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f}")

    # ==========================================
    #  평가 부분 (validation)
    # ==========================================
    model.eval()
    eval_loss = 0.0
    preds_list = []
    labels_list = []

    with torch.no_grad():
        for batch in eval_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            logits = out.logits

            labels_for_crf = batch["labels"].clone()
            labels_for_crf[labels_for_crf == -100] = 0
            mask = batch["attention_mask"].bool()

            # CRF 손실
            loss = -crf(logits, labels_for_crf, mask=mask, reduction="mean")
            eval_loss += loss.item()

            # CRF 디코딩 (배치마다 시퀀스 길이만큼 라벨 예측)
            decoded = crf.decode(logits, mask=mask)

            # 텐서 -> 리스트
            for i, d in enumerate(decoded):
                true_labels = labels_for_crf[i][mask[i]].tolist()
                preds_list.extend(d)
                labels_list.extend(true_labels)

    avg_eval_loss = eval_loss / len(eval_dataloader)
    print(f"[Epoch {epoch+1}] Eval Loss: {avg_eval_loss:.4f}")

    # ==========================================
    # classification report 출력
    # ==========================================
    label_names = [id2label[i] for i in range(len(id2label))]
    print(classification_report(labels_list, preds_list, target_names=label_names, zero_division=0))
